# Notebook Overview

**Purpose:**
This notebook performs multi-objective hyperparameter optimisation for sequence forecasting models (LSTM or xLSTM) using Optuna. It trains candidate model configurations on a fixed train/validation split, optimises validation RMSE while simultaneously maximising validation R², and then evaluates a selected Pareto-optimal configuration on the held-out test set with denormalised metrics and diagnostic plots.

**Inputs:**
- Time series CSV (configurable): `../data/ts_with_sentiment/AAPL_ts_with_sentiment.csv`
- Feature specification (e.g., price-only or price plus sentiment) defined via `FEATURE_COLS` and `TARGET_COL`.
- Preprocessing and split parameters: `TRAIN_SPLIT`, `VAL_SPLIT`, `WINDOW_SIZE`, `HORIZON`, `STRIDE`, `NORMALISE`, `NORM_METHOD`.
- Optuna storage (SQLite): `../data/optuna_studies/BachelorThesis.db`
- Local project modules for preprocessing and modelling:
  - `src.data_prep.PrepAndDataLoader`
  - `src.model_wrapper.Model`

**Outputs:**
- Persisted Optuna study with multi-objective trials (RMSE minimisation, R² maximisation) stored in the configured SQLite database.
- Trial artifacts saved during training:
  - Model checkpoints in `../data/optuna_checkpoints/{MODEL_TYPE}_optuna_H{HORIZON}_F{n_features}/`
  - Training-history JSON files associated with checkpoints (loss and validation loss).
- Console summary of Pareto-optimal trials and the selected compromise trial.
- Optuna visualisations of hyperparameter importance for both objectives (RMSE and R²).
- Final test-set evaluation of the selected model:
  - Scaled and denormalised RMSE/R²
  - Training-history plot for the selected trial
  - True vs. predicted plot on the test set (typically for the final horizon step t+H).

**Process Summary:**
The notebook first constructs train/validation/test tensors using a consistent preprocessing pipeline and normalisation settings, caching base values required for denormalisation when percentage normalisation is used. During optimisation, each Optuna trial samples a model configuration, trains the model with early stopping on validation loss, and computes validation RMSE and R² by flattening across all samples and horizons. The study is run as a multi-objective optimisation problem, yielding a Pareto set of non-dominated trials. A single compromise trial is selected by prioritising minimal RMSE and using R² as a tie-breaker. Finally, the selected checkpoint is restored and evaluated once on the test set, with optional denormalisation back to the original price scale and corresponding visual diagnostics.


In [ ]:
%load_ext autoreload
%autoreload 2
# =========================
# Imports
# =========================
import os
from typing import Dict, Any
from pathlib import Path
from typing import Literal
import json

import numpy as np
import torch
import optuna
from optuna.visualization import plot_param_importances
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error

import matplotlib.pyplot as plt
import pandas as pd

from src.model_wrapper import Model, set_global_seed
from src.data_prep import PrepAndDataLoader

optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Info] Using device: {DEVICE}")

In [ ]:
# =========================
# Helper
# =========================

def plot_history_from_json(history_json_path: str, title: str = ""):
    """Plot train/val loss aus einer JSON-History-Datei (wie im Model.train gespeichert)."""
    if not os.path.isfile(history_json_path):
        print(f"[WARN] History file not found: {history_json_path}")
        return
    with open(history_json_path, "r") as f:
        hist = json.load(f)

    plt.figure(figsize=(8, 4))
    if "loss" in hist:
        plt.plot(hist["loss"], label="train_loss")
    if "val_loss" in hist:
        plt.plot(hist["val_loss"], label="val_loss")
    plt.title(f"Training History {title}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# =========================
# Variables (edit this cell)
# =========================

# --- Data settings ---
DATA_FILE: str = "../data/ts_with_sentiment/AAPL_ts_with_sentiment.csv"  # path to your CSV
#FEATURE_COLS = ["Close", "pos_mean", "neut_mean", "neg_mean", "news_count"]
FEATURE_COLS = ["Close"]
TARGET_COL: str = "Close"                           # target column name
TRAIN_SPLIT: float = 0.6
VAL_SPLIT: float = 0.2

# --- Window / horizon settings ---
WINDOW_SIZE: int = 60           # input sequence length
HORIZON: int = 3                # prediction range (direct multi-step)
STRIDE: int = 1

# --- Normalization settings ---
NORMALISE: bool = True
NORM_METHOD: Literal["percentage"] = "percentage"     # "minmax" or "percentage"

# --- Training settings ---
EPOCHS: int = 100
BATCH_SIZE: int = 128
PATIENCE: int = 10
VERBOSE_TRAIN: int = 1

# --- Model type ---
MODEL_TYPE: str = "LSTM"  # xLSTM or LSTM

# --- Optuna settings ---
N_TRIALS: int = 180
STUDY_NAME: str = f"{MODEL_TYPE}_H{HORIZON}_F{len(FEATURE_COLS)}"
CHECKPOINT_DIR: str = f"../data/optuna_checkpoints/{MODEL_TYPE}_optuna_H{HORIZON}_F{len(FEATURE_COLS)}"

DB_PATH= Path(f"../data/optuna_studies/BachelorThesis.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
STORAGE_URL = f"sqlite:///{DB_PATH.as_posix()}"

# --- Seed ---
GLOBAL_SEED: int = 42

In [ ]:
# =========================
# Data preparation
# =========================

set_global_seed(GLOBAL_SEED)

print("[Data] Initializing PrepAndDataLoader...")
prep = PrepAndDataLoader(
    filename=DATA_FILE,
    training_split=TRAIN_SPLIT,
    validation_split=VAL_SPLIT,
    cols=FEATURE_COLS,
    target_col=TARGET_COL,
)

tmp_model = Model()
X_tr, y_tr, X_va, y_va, X_te, y_te = tmp_model.prepare_data_from_prep(
    prep,
    normalise=NORMALISE,
    window_size=WINDOW_SIZE,
    prediction_range=HORIZON,
    norm_method=NORM_METHOD,
    stride=STRIDE,
    verbose=1,
)
dates_te = prep.get_prediction_dates(
    "test",
    window_size=WINDOW_SIZE,
    prediction_range=HORIZON,
    stride=STRIDE,
)  # Form [N, HORIZON]
bT_te = tmp_model._cache["baseT_te"]  # Shape [N, 1]
N_FEATURES = X_tr.shape[-1]
print(f"[Data] Number of features: {N_FEATURES}")


In [ ]:
# =========================
# Hyperparameter builders
# =========================

def build_xlstm_config(trial: optuna.Trial) -> Dict[str, Any]:
    """
    Build a configuration dictionary for the xLSTM model from an Optuna trial.
    This config is consumed by Model.build(...).

    - units: shared embedding dimension / hidden size
    - mLSTM and sLSTM have independent num_heads (both must divide `units`)
    - Conv1d kernel sizes never use 1 (0 = no conv, >=2 = valid)
    """

    # -------------------------
    # Core model size & layout
    # -------------------------
    units = trial.suggest_categorical("units", [32, 48, 64, 96, 128])

    xlstm_num_blocks = trial.suggest_categorical(
        "xlstm_num_blocks",
        [2, 4, 6, 8],
    )

    # Mix of mLSTM and sLSTM blocks
    slstm_mode = trial.suggest_categorical(
        "slstm_mode",
        ["m_only", "mixed"],
    )
    if slstm_mode == "m_only":
        xlstm_slstm_at = []  # no sLSTM blocks
    else:
        # simple variant: last block is sLSTM
        xlstm_slstm_at = [xlstm_num_blocks - 1]

    # -------------------------
    # Regularization & optimizer
    # -------------------------
    dropout = trial.suggest_categorical(
        "dropout",
        [0.0, 0.1, 0.2],
    )

    optimizer = trial.suggest_categorical(
        "optimizer",
        ["adamW", "adam"],
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        3e-3,
        log=True,
    )

    # -------------------------
    # mLSTM-specific hyperparameters
    # -------------------------

    # Conv1d kernel: 0 = no conv, >=2 = valid conv (1 würde zu S=0 führen und crashen)
    mlstm_conv1d_kernel_size = trial.suggest_categorical(
        "xlstm_mlstm_conv1d_kernel_size",
        [2, 3, 4],
    )

    mlstm_qkv_proj_blocksize = trial.suggest_categorical(
        "xlstm_mlstm_qkv_proj_blocksize",
        [2, 4],
    )

    mlstm_proj_factor = trial.suggest_categorical(
        "xlstm_mlstm_proj_factor",
        [1.0, 1.5, 2.0],
    )

    # Separate mLSTM heads; must divide `units`
    mlstm_head_candidates = [h for h in [1, 2, 4, 8] if units % h == 0]
    if not mlstm_head_candidates:
        mlstm_head_candidates = [1]

    mlstm_num_heads = trial.suggest_categorical(
        "xlstm_mlstm_num_heads",
        mlstm_head_candidates,
    )

    # -------------------------
    # sLSTM-specific hyperparameters
    # -------------------------

    # Conv1d kernel: again no 1 (0 = off, >=2 = ok)
    slstm_conv1d_kernel = trial.suggest_categorical(
        "xlstm_slstm_conv1d_kernel",
        [2, 3, 4],
    )

    # Separate sLSTM heads; also must divide `units`
    slstm_head_candidates = [h for h in [1, 2, 4, 8] if units % h == 0]
    if not slstm_head_candidates:
        slstm_head_candidates = [1]

    slstm_num_heads = trial.suggest_categorical(
        "xlstm_slstm_num_heads",
        slstm_head_candidates,
    )

    # -------------------------
    # Final config dict
    # -------------------------
    config: Dict[str, Any] = {
        # data / task
        "window_size": WINDOW_SIZE,
        "n_features": N_FEATURES,
        "horizon": HORIZON,

        # shared model hyperparameters
        "units": int(units),
        "dropout": float(dropout),
        "optimizer": optimizer,
        "learning_rate": float(learning_rate),
        "loss": "mse",

        # xLSTM block stack
        "xlstm_num_blocks": int(xlstm_num_blocks),
        "xlstm_slstm_at": xlstm_slstm_at,

        # mLSTM branch
        "xlstm_mlstm_conv1d_kernel_size": int(mlstm_conv1d_kernel_size),
        "xlstm_mlstm_qkv_proj_blocksize": int(mlstm_qkv_proj_blocksize),
        "xlstm_mlstm_num_heads": int(mlstm_num_heads),
        "xlstm_mlstm_proj_factor": float(mlstm_proj_factor),

        # sLSTM branch
        "xlstm_slstm_conv1d_kernel": int(slstm_conv1d_kernel),
        "xlstm_slstm_num_heads": int(slstm_num_heads),
    }

    return config


def build_lstm_config(trial: optuna.Trial) -> Dict[str, Any]:
    """
    Placeholder for a plain LSTM config.
    Currently not used, but ready for future extension.
    """
    units = trial.suggest_categorical("units", [32, 64, 128])
    lstm_layers = trial.suggest_int("lstm_layers", 1, 4)
    dropout = trial.suggest_categorical("dropout", [0.0, 0.1, 0.2])
    optimizer = trial.suggest_categorical("optimizer", ["adamW", "adam"])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True)

    return {
        "window_size": WINDOW_SIZE,
        "n_features": N_FEATURES,
        "horizon": HORIZON,
        "units": int(units),
        "dropout": float(dropout),
        "optimizer": optimizer,
        "learning_rate": float(learning_rate),
        "loss": "mse",
        "lstm_layers": int(lstm_layers),
    }


In [ ]:
# =========================
# Optuna objective (RMSE optimize, R² log)
# =========================

def objective(trial: optuna.Trial) -> float:
    """
    Optuna objective function.

    - Trains an xLSTM model with hyperparameters sampled from the trial.
    - Uses Train set for learning.
    - Uses Validation set for metric computation.
    - Returns RMSE on the validation set as objective.
    - Logs R² as additional metric in trial.user_attrs.
    """

    # seed for reproducibility
    set_global_seed(GLOBAL_SEED)

    # --- Build model config ---
    if MODEL_TYPE == "xLSTM":
        config = build_xlstm_config(trial)
    elif MODEL_TYPE == "LSTM":
        config = build_lstm_config(trial)
    else:
        raise ValueError(f"Unsupported MODEL_TYPE: {MODEL_TYPE}")

    # --- Initialize model wrapper ---
    model = Model()
    model.build(config=config, model_type=MODEL_TYPE)

    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # --- Train on train set, validate on validation set ---
    best_ckpt_path = model.train(
        X_tr,
        y_tr,
        X_va,
        y_va,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        save_dir=CHECKPOINT_DIR,
        patience=PATIENCE,
        monitor="val_loss",
        verbose=VERBOSE_TRAIN,
        restore_best_weights=True,
        save_best_only=True,
    )

    y_val_true = y_va.squeeze(-1)              # [N, H]
    y_val_pred = model.predict_multi_horizon(  # [N, H]
        X_va,
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    # Flatten over all horizons and samples
    y_true_flat = y_val_true.reshape(-1)
    y_pred_flat = y_val_pred.reshape(-1)

    val_mse = mean_squared_error(y_true_flat, y_pred_flat)
    val_rmse = float(np.sqrt(val_mse))
    val_r2 = float(r2_score(y_true_flat, y_pred_flat))

    # Store paths, config and additional metrics in the trial for later inspection
    trial.set_user_attr("best_checkpoint_path", best_ckpt_path)
    trial.set_user_attr("config", config)
    trial.set_user_attr("val_mse", val_mse)
    trial.set_user_attr("val_rmse", val_rmse)
    trial.set_user_attr("val_r2", val_r2)

    # Optuna optimizes this value
    return val_rmse, val_r2

In [ ]:
# =========================
# Run Optuna study
# =========================

print(f"[Optuna] Starting study '{STUDY_NAME}' with {N_TRIALS} trials...")

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_URL,
    directions=["minimize", "maximize"],  # RMSE, R²
    load_if_exists=True,
)

study.optimize(objective, n_trials=N_TRIALS)

In [ ]:
print("[Optuna] Study finished.")

# Pareto-Menge holen (Multi-Objective!)
pareto_trials = study.best_trials
print(f"[Optuna] Number of Pareto-optimal trials: {len(pareto_trials)}")

best_trial = min(pareto_trials, key=lambda t: (t.values[0], -t.values[1]))

print("  Selected compromise trial (RMSE min, R² max given RMSE):")
print(f"    val_RMSE = {best_trial.values[0]:.6f}")
print(f"    val_R²   = {best_trial.values[1]:.4f}")
print("  Best hyperparameters:")
for k, v in best_trial.params.items():
    print(f"    {k}: {v}")

best_ckpt = best_trial.user_attrs.get("best_checkpoint_path", None)
best_val_r2 = best_trial.user_attrs.get("val_r2", None)
if best_ckpt is not None:
    print(f"[Optuna] Best checkpoint path: {best_ckpt}")
if best_val_r2 is not None:
    print(f"[Optuna] Best trial validation R² (stored): {best_val_r2:.4f}")

In [ ]:
print("Hyperparameter Importance für RMSE:")
plot_param_importances(study, target=lambda t: t.values[0], target_name="RMSE").show()

print("Hyperparameter Importance für R²:")
plot_param_importances(study, target=lambda t: t.values[1], target_name="R²").show()

In [ ]:
# =========================
# Final evaluation on test set (only once, incl. plots)
# =========================

bT_te = tmp_model._cache["baseT_te"]  # [N, 1]

if best_ckpt is not None:
    print("[Eval] Loading best model and evaluating on test set...")

    # Konfiguration des ausgewählten Pareto-Trials
    best_config = best_trial.user_attrs["config"]

    eval_model = Model()
    eval_model.build(config=best_config, model_type=MODEL_TYPE)
    eval_model.load(best_ckpt)

    # ---------------------------------
    # 1) Training-History dieses Trials
    # ---------------------------------
    history_json = best_ckpt.replace(".pt", "_history.json")
    plot_history_from_json(
        history_json,
        title=f"H={HORIZON}, {os.path.basename(DATA_FILE)}"
    )

    # ---------------------------------
    # 2) Vorhersagen im Skalenraum
    # ---------------------------------
    y_test_true_scaled = y_te.squeeze(-1)  # [N, HORIZON]
    y_test_pred_scaled = eval_model.predict_multi_horizon(
        X_te,
        batch_size=BATCH_SIZE,
        verbose=0,
    )  # [N, HORIZON]

    # Flatten über alle Horizonte und Samples
    y_true_flat_scaled = y_test_true_scaled.reshape(-1)
    y_pred_flat_scaled = y_test_pred_scaled.reshape(-1)

    test_rmse_scaled = float(
        root_mean_squared_error(y_true_flat_scaled, y_pred_flat_scaled)
    )
    test_r2_scaled = float(r2_score(y_true_flat_scaled, y_pred_flat_scaled))

    print(f"[Eval - SCALED] RMSE={test_rmse_scaled:.6f} | R²={test_r2_scaled:.4f}")

    # ---------------------------------
    # 3) Denormalisation (Original-Skala)
    # ---------------------------------
    if NORMALISE:
        # Auf [N, H, 1] bringen
        yhat_scaled_3d = y_test_pred_scaled[..., None]  # [N, H, 1]
        ytrue_scaled_3d = y_te  # [N, H, 1]

        if NORM_METHOD == "percentage":
            yhat_level = prep.denormalise(
                yhat_scaled_3d,
                method="percentage",
                base_values=bT_te,
                normalise=True,
            ).squeeze(-1)  # [N, H]

            ytrue_level = prep.denormalise(
                ytrue_scaled_3d,
                method="percentage",
                base_values=bT_te,
                normalise=True,
            ).squeeze(-1)

        elif NORM_METHOD == "minmax":
            # minmax ignoriert base_values
            yhat_level = prep.denormalise(
                yhat_scaled_3d,
                method="minmax",
                base_values=None,
                normalise=True,
            ).squeeze(-1)

            ytrue_level = prep.denormalise(
                ytrue_scaled_3d,
                method="minmax",
                base_values=None,
                normalise=True,
            ).squeeze(-1)

        else:
            raise ValueError(f"Unknown normalization method: {NORM_METHOD}")

        # RMSE & R² im Original-Level
        y_true_flat_level = ytrue_level.reshape(-1)
        y_pred_flat_level = yhat_level.reshape(-1)

        test_rmse_level = float(
            root_mean_squared_error(y_true_flat_level, y_pred_flat_level)
        )
        test_r2_level = float(r2_score(y_true_flat_level, y_pred_flat_level))

        print(f"[Eval - LEVEL]  RMSE={test_rmse_level:.6f} | R²={test_r2_level:.4f}")

    else:
        # Keine Normalisierung → scaled == level
        yhat_level = y_test_pred_scaled
        ytrue_level = y_test_true_scaled
        print("[Eval] NORMALISE=False → scaled values are already in original scale.")

    # ---------------------------------
    # 4) Plot: True vs. Pred (Originalskala)
    # ---------------------------------
    if STRIDE == 1:
        # Nur den letzten Horizont t+HORIZON plotten
        h_idx = HORIZON - 1  # Index des letzten Horizonts

        dates_h = dates_te[:, h_idx]  # [N]
        true_h = ytrue_level[:, h_idx]  # [N]
        pred_h = yhat_level[:, h_idx]  # [N]

        plt.figure(figsize=(12, 5))
        plt.plot(dates_h, true_h, label=f"True (t+{HORIZON})")
        plt.plot(dates_h, pred_h, label=f"Pred (t+{HORIZON})")

        plt.title(f"{os.path.basename(DATA_FILE)} | Horizon = t+{HORIZON}")
        plt.xlabel("Date")
        plt.ylabel(TARGET_COL)
        plt.legend()
        plt.grid(True)
        plt.gcf().autofmt_xdate()
        plt.show()

    else:
        # Alle Horizonte zu einer einzigen Zeitreihe mergen (z. B. bei STRIDE>1)
        dates_flat = dates_te.reshape(-1)
        true_flat = ytrue_level.reshape(-1)
        pred_flat = yhat_level.reshape(-1)

        df_plot = pd.DataFrame(
            {
                "date": dates_flat,
                "true": true_flat,
                "pred": pred_flat,
            }
        )
